In [ ]:
import numpy as np
import pickle
import codecs
from collections import defaultdict
from scipy.stats import spearmanr
from sklearn import preprocessing
import pickle
from collections import defaultdict
final_dict=defaultdict(list)
# from gensim.models import Word2Vec
# import gensim.downloader
# from glove import Corpus, Glove
# from gensim.models import FastText
from scipy.stats import spearmanr
from sklearn import preprocessing
from sklearn.metrics.pairwise import cosine_similarity
from scipy import spatial
from scipy.stats import kendalltau
from contextlib import redirect_stdout

# model= Word2Vec.load("word2vec_billion.model")
# model_fast= FastText.load("billion_fasttext.model")
# model_glove= Glove.load("billion_glove_updated.model")


folder_path = 'datasets'
dir_count = 0
for item in os.listdir(folder_path):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        dir_count += 1
progress_bar = tqdm(total=dir_count, desc="Running Datasets")

for folder_name in os.listdir(folder_path):
    current_folder_path = os.path.join(folder_path, folder_name)
    if os.path.isdir(current_folder_path):
        files_start_name = os.path.join(current_folder_path, folder_name)
        dict_path = files_start_name + '_profile_dict.pkl'
        if os.path.exists(dict_path) == False:
            progress_bar.update(1)
            continue
        saved= open(dict_path, "rb")
        profile_dict_word = pickle.load(saved)
        saved.close()

        ds_path = files_start_name + '.csv'
        if os.path.exists(ds_path) == False:
            progress_bar.update(1)
            continue
        fread_simlex=codecs.open(ds_path, 'r', 'utf-8')
        pair_list = []
        line_number = 0
        for line in fread_simlex:
            if line_number > 0:
                tokens = line.split()
                word_i = tokens[0].lower()
                word_j = tokens[1].lower()
                score = float(tokens[2])
                pair_list.append( ((word_i, word_j), score) )
            line_number += 1

        calculated_score=[]
        extracted_list = []
        original_score=[]
        # calculated_w2v=[]
        word_pairs=[]
        # calculated_fast=[]
        # calculated_glove=[]
        words = (model.wv.index_to_key)
        embedding_weight_w2v=[]
        from sklearn.metrics.pairwise import cosine_similarity
        for (x,y) in pair_list:
            if x in profile_dict_word:
                word1, word2=x
                if word1 in words and word2 in words:
                    word1_prof=profile_dict_word[x]
                    # w2v_score=model.wv.similarity(word1, word2)
                    # fast_score=model_fast.wv.similarity(word1, word2)
                    # glove_vec1= [model_glove.word_vectors[model_glove.dictionary[word1]]]
                    # glove_vec2= [model_glove.word_vectors[model_glove.dictionary[word2]]]
                    # glove_score= 1-spatial.distance.cosine(glove_vec1,glove_vec2)
                    extracted_list.append((x, word1_prof))
                    calculated_score.append(word1_prof)
                    original_score.append(y)
                    # calculated_w2v.append(w2v_score)
                    word_pairs.append(x)
                    # calculated_fast.append(fast_score)
                    # embedding_weight_w2v.append(model.wv[word1])
                    # embedding_weight_w2v.append(model.wv[word2])
                    # calculated_glove.append(glove_score)

        with open(files_start_name + '_evaluation.txt', 'w') as file, redirect_stdout(file):
            print(f'original score {len(original_score)} calculated w2v {len(calculated_score)} calculated fast {len(calculated_fast)} Glove {len(calculated_glove)}')
            # spearman_w2v = spearmanr(original_score, calculated_w2v)
            # spearman_fast = spearmanr(original_score, calculated_fast)
            spearman_TM = spearmanr(original_score, calculated_score)
            # spearman_glove = spearmanr(original_score, calculated_glove)
            # spearman_tm_w2v= spearmanr(calculated_glove, calculated_w2v)
            # spearman_w2v = round(spearman_w2v[0], 3)
            # spearman_fast = round(spearman_fast[0], 3)
            spearman_TM = round(spearman_TM[0], 3)
            # spearman_glove = round(spearman_glove[0], 3)
            # spearman_tm_w2v= round(spearman_tm_w2v[0], 3)
            print(f'spearman TM {spearman_TM}')

            total_list=[]

            total_list.append(original_score)
            total_list.append(calculated_score)

            # total_w2v=[]
            # total_w2v.append(original_score)
            # total_w2v.append(calculated_w2v)

            # total_fast=[]
            # total_fast.append(original_score)
            # total_fast.append(calculated_fast)

            # tm_w2v=[]
            # tm_w2v.append(calculated_score)
            # tm_w2v.append(calculated_w2v)

            # tm_glove=[]
            # tm_glove.append(original_score)
            # tm_glove.append(calculated_glove)

            similarity = cosine_similarity(total_list)
            similarity_w2v= cosine_similarity(total_w2v)
            similarity_fast= cosine_similarity(total_fast)
            similarity_glove= cosine_similarity(tm_glove)
            similarity_tm_w2v= cosine_similarity(tm_w2v)
            print(f'Cosine TM {similarity} Consine w2v {similarity_w2v}  Cosine Fast {similarity_fast} Cosine tm_w2v {similarity_tm_w2v} Cosine glove{similarity_glove}')

            TM_corr= np.corrcoef(original_score, calculated_score)
            w2v_corr= np.corrcoef(original_score, calculated_w2v)
            fast_corr= np.corrcoef(original_score, calculated_fast)
            glove_corr= np.corrcoef(original_score, calculated_glove)
            print(f'pearson TM {TM_corr} pearson w2v {w2v_corr} pearson fast {fast_corr} pearson glove{glove_corr}')

            kendal_TM, _ = kendalltau(original_score, calculated_score)
            kendal_w2v, _ = kendalltau(original_score, calculated_w2v)
            kendal_fast, _ = kendalltau(original_score, calculated_fast)
            kendal_glove, _ = kendalltau(original_score, calculated_glove)
            print(f'kendal TM {kendal_TM} kendal W2V {kendal_w2v} Kendal fast {kendal_fast} kendal glove{kendal_glove}')
    progress_bar.update(1)
progress_bar.close()